# GRPO for Agentic LLM Fine-Tuning: Reproducibility Notebook

**Paper:** *Group Relative Policy Optimization for Agentic LLM Fine-Tuning*
**Affiliation:** PES University MTech Capstone (Group 6)
**Repository:** [pes-llm-research/tinker-rl-lab](https://github.com/pes-llm-research/tinker-rl-lab)

---

This notebook provides **full reproducibility** for the key results reported in the paper:

| Section | What | Runtime |
|---------|------|--------|
| A | Install + Setup | ~4 min |
| B | GRPO Training (Qwen3-8B on GSM8K) | ~35 min (A100) |
| C | Held-Out GSM8K Evaluation (200 examples) | ~15 min |
| D | Results Visualization & Statistics | ~1 min |

**Requirements:** Colab Pro with A100 GPU (or Vast.ai A100 instance)

> **Note:** Original experiments used the Tinker cloud training API. This notebook uses Unsloth + TRL GRPOTrainer as a local drop-in replacement that produces equivalent results.

## A. Setup & Installation

In [ ]:
# A1. Verify GPU
!nvidia-smi | head -5
import torch
print(f'\nPyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT AVAILABLE"}')
assert torch.cuda.is_available(), 'GPU required — switch to A100 runtime'

In [ ]:
# A2. Install dependencies
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q 'trl>=0.9.0' 'peft>=0.12.0' 'accelerate>=0.33.0'
!pip install -q math-verify latex2sympy2-extended
!pip install -q datasets wandb pyyaml
print('\n--- Dependencies installed ---')

In [ ]:
# A3. Clone repository
import os

REPO_URL = 'https://github.com/pes-llm-research/tinker-rl-lab.git'
REPO_DIR = '/content/tinker-rl-lab'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --rebase

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# A4. Credentials (edit before running)
import os

# HuggingFace token — required for gated models
HF_TOKEN = 'hf_YOUR_TOKEN_HERE'  # Get at https://huggingface.co/settings/tokens

# WandB — optional but recommended for metric tracking
WANDB_API_KEY = ''  # Get at https://wandb.ai/authorize (leave empty to skip)

os.environ['HF_TOKEN'] = HF_TOKEN
if WANDB_API_KEY:
    os.environ['WANDB_API_KEY'] = WANDB_API_KEY
    import wandb
    wandb.login(key=WANDB_API_KEY)
else:
    os.environ['WANDB_DISABLED'] = 'true'
    print('WandB disabled — metrics will be saved locally only')

## B. GRPO Training

Trains Qwen3-8B on GSM8K using GRPO with LoRA rank 32.

| Hyperparameter | Value |
|---|---|
| Model | Qwen/Qwen3-8B |
| LoRA rank | 32 |
| Learning rate | 4e-5 |
| Group size | 16 |
| Batch size | 128 |
| Steps | 50 |
| Max token length | 2048 |

In [ ]:
# B1. Run GRPO training
# This takes ~35 minutes on A100 40GB

SEED = 42
CONFIG = 'atropos/configs/gsm8k_qwen_8b.yaml'

!cd atropos && python train_grpo_unsloth.py \
    --config configs/gsm8k_qwen_8b.yaml \
    --seed {SEED}

In [ ]:
# B2. Training results
import csv, os
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

CHECKPOINT_DIR = 'atropos/checkpoints/gsm8k_qwen8b/'
csv_path = os.path.join(CHECKPOINT_DIR, 'reward_log.csv')

if os.path.exists(csv_path):
    steps, rewards = [], []
    with open(csv_path) as f:
        for row in csv.DictReader(f):
            steps.append(int(row['step']))
            rewards.append(float(row['mean_reward']))

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(steps, rewards, color='#2196F3', linewidth=2)
    ax.fill_between(steps, rewards, alpha=0.1, color='#2196F3')
    ax.set_xlabel('Training Step')
    ax.set_ylabel('Mean Reward (Accuracy)')
    ax.set_title(f'GRPO Training Curve — Qwen3-8B on GSM8K (seed={SEED})')
    ax.set_ylim(-0.05, 1.1)
    ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.3)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    print(f'Initial reward: {rewards[0]:.4f}')
    print(f'Final reward:   {rewards[-1]:.4f}')
    print(f'Mean (last 10): {sum(rewards[-10:])/10:.4f}')
else:
    print(f'No reward log found at {csv_path}')
    print('If using pre-trained checkpoint, skip to Section C.')

## C. Held-Out GSM8K Evaluation

Evaluates the trained checkpoint on the **held-out GSM8K test set** (1,319 examples).

This is the key result reported in the paper: generalization accuracy on unseen problems,
not training reward.

In [ ]:
# C1. Run held-out evaluation
# Uses greedy decoding (temperature=0) for reproducibility
# ~15 min for 200 examples, ~90 min for full 1319

CHECKPOINT = 'atropos/checkpoints/gsm8k_qwen8b/'  # Local checkpoint from training
N_EXAMPLES = 200  # Set to None for full test set (1319 examples)
SEED = 42

limit_flag = f'--limit {N_EXAMPLES}' if N_EXAMPLES else ''

!cd reports/final && python evaluate_gsm8k_test.py \
    --checkpoint_path ../../{CHECKPOINT} \
    --model_name Qwen/Qwen3-8B \
    --seed {SEED} \
    --max_tokens 2048 \
    --output gsm8k_heldout_colab.json \
    {limit_flag}

In [ ]:
# C2. Display evaluation results
import json

RESULT_FILE = 'reports/final/gsm8k_heldout_colab.json'

with open(RESULT_FILE) as f:
    results = json.load(f)

s = results['summary']
c = results['config']

print('=' * 55)
print('  GSM8K HELD-OUT TEST SET EVALUATION')
print('=' * 55)
print(f'  Model:       {c["model"]}')
print(f'  Test size:   {c["test_size"]} examples')
print(f'  Seed:        {c["seed"]}')
print(f'  Temperature: {c["temperature"]} (greedy)')
print('-' * 55)
print(f'  Correct:     {s["correct"]}')
print(f'  Incorrect:   {s["incorrect"]}')
print(f'  Errors:      {s["errors"]}')
print(f'  Accuracy:    {s["accuracy_percent"]}')
ci = s.get('accuracy_ci_95_percent', ['N/A', 'N/A'])
print(f'  95% CI:      [{ci[0]}, {ci[1]}]')
print('=' * 55)

## D. Results Visualization & Paper Figures

In [ ]:
# D1. Error analysis — what types of problems does the model get wrong?
import json

with open('reports/final/gsm8k_heldout_colab.json') as f:
    results = json.load(f)

incorrect = [e for e in results['examples'] if e.get('status') == 'incorrect']
errors = [e for e in results['examples'] if e.get('status') == 'error']
correct = [e for e in results['examples'] if e.get('status') == 'correct']

print(f'Correct:   {len(correct)}')
print(f'Incorrect: {len(incorrect)}')
print(f'Errors:    {len(errors)}')
print()

if incorrect:
    print('--- Sample Incorrect Predictions ---')
    for ex in incorrect[:5]:
        print(f'  Q: {ex.get("question", "?")}')
        print(f'  Predicted: {ex.get("predicted")}  |  Ground Truth: {ex.get("ground_truth")}')
        print()

In [ ]:
# D2. Comparison: Base model vs GRPO-trained
import matplotlib.pyplot as plt
import numpy as np

# Pre-computed base model accuracy (from gsm8k_base_smoke_results.json)
base_acc = 0.875  # Qwen3-8B base on 40-example sample
base_ci = [0.775, 0.975]

# Trained model — fill from evaluation above
with open('reports/final/gsm8k_heldout_colab.json') as f:
    r = json.load(f)
trained_acc = r['summary']['accuracy']
trained_ci = r['summary'].get('accuracy_ci_95', [trained_acc, trained_acc])

fig, ax = plt.subplots(figsize=(8, 5))
models = ['Qwen3-8B\n(Base)', 'Qwen3-8B\n(+GRPO LoRA)']
accs = [base_acc, trained_acc]
ci_lower = [base_acc - base_ci[0], trained_acc - trained_ci[0]]
ci_upper = [base_ci[1] - base_acc, trained_ci[1] - trained_acc]

colors = ['#95a5a6', '#2ecc71']
bars = ax.bar(models, accs, color=colors, width=0.5,
              yerr=[ci_lower, ci_upper], capsize=8, error_kw={'linewidth': 2})

for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
            f'{acc:.1%}', ha='center', va='bottom', fontweight='bold', fontsize=14)

ax.set_ylabel('GSM8K Test Accuracy', fontsize=13)
ax.set_title('GRPO Fine-Tuning Impact on GSM8K', fontsize=14, fontweight='bold')
ax.set_ylim(0, 1.15)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('reports/final/fig_base_vs_grpo.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# D3. Multi-seed summary (if multiple seed results are available)
import glob, json, statistics

files = sorted(glob.glob('reports/final/gsm8k_heldout_seed*.json'))
if files:
    accs = []
    print('=== Multi-Seed Held-Out Results ===')
    print(f'{"File":<35s} {"Accuracy":>10s} {"CI 95%":>20s}')
    print('-' * 68)
    for f in files:
        d = json.load(open(f))
        s = d['summary']
        acc = s['accuracy']
        ci = s.get('accuracy_ci_95_percent', ['?', '?'])
        accs.append(acc)
        fname = os.path.basename(f)
        print(f'{fname:<35s} {acc:>9.1%}   [{ci[0]:>6s}, {ci[1]:>6s}]')

    if len(accs) > 1:
        mean = statistics.mean(accs)
        std = statistics.stdev(accs)
        print('-' * 68)
        print(f'{"MEAN ± STD":<35s} {mean:>9.1%} ± {std:.1%}')
        print(f'\nPaper-ready: {mean:.1%} ± {std:.1%} ({len(accs)} seeds, n={d["config"]["test_size"]} each)')
else:
    print('No multi-seed results found. Run Section C with different seeds.')

## E. Pre-Computed Results (Paper Reference)

The following cells display the pre-computed results from the paper's original Tinker API runs,
for comparison with the locally reproduced results above.

In [ ]:
# E1. Original training curves from Tinker API runs
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

# Original Tinker results — Qwen3-8B seed 42
original_steps = list(range(50))
original_rewards = [
    0.0703, 0.0078, 0.0391, 0.0391, 0.0703, 0.0469, 0.0625, 0.0938, 0.0, 0.0391,
    0.0391, 0.0391, 0.1094, 0.1562, 0.1094, 0.1484, 0.1172, 0.1328, 0.0859, 0.0469,
    0.1172, 0.1719, 0.125, 0.3359, 0.2734, 0.75, 0.9375, 0.5078, 0.4062, 0.6641,
    1.0, 0.7969, 0.5469, 0.9844, 1.0, 0.8672, 1.0, 0.8047, 0.8672, 0.9922,
    0.9922, 0.7344, 1.0, 0.8828, 1.0, 1.0, 0.9844, 0.8438, 1.0, 1.0
]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(original_steps, original_rewards, color='#2ecc71', linewidth=2, alpha=0.8, label='Seed 42 (Tinker)')
ax.fill_between(original_steps, original_rewards, alpha=0.1, color='#2ecc71')
ax.set_xlabel('Training Step', fontsize=12)
ax.set_ylabel('Mean Reward (Accuracy)', fontsize=12)
ax.set_title('GRPO Training: Qwen3-8B on GSM8K (Original Tinker Run)', fontsize=14, fontweight='bold')
ax.set_ylim(-0.05, 1.1)
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.3, label='Perfect')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

print(f'Training reward: {original_rewards[0]:.1%} -> {original_rewards[-1]:.1%}')
print(f'Mean last 10: {sum(original_rewards[-10:])/10:.1%}')

## F. Architecture Diagram

In [ ]:
print("""
GRPO Training Pipeline
======================

  +-------------------+     +-------------------+     +-------------------+
  |   Atropos          |     |   Environment     |     |   Trainer         |
  |   (Coordinator)    |---->|   (GSM8K/MATH)    |---->|   (GRPO + LoRA)   |
  |   run-api          |<----|   Binary rewards   |<----|   Cloud/Local GPU |
  +-------------------+     +-------------------+     +-------------------+
         |                         |                         |
         |--- Manages rollouts     |--- Scores answers       |--- Updates LoRA
         |--- Batches groups       |--- 0/1 per completion   |--- Importance sampling
         |--- Tracks metrics       |--- Group-relative       |--- Checkpoint save
         |                         |                         |
         +-------------------------+-------------------------+
                           Data flow loop

Key GRPO Innovation:
  - No critic/value network needed (unlike PPO)
  - Advantages computed relative to group mean
  - Group size G=16: generate 16 completions per prompt
  - Better completions get positive advantage, worse get negative
  - LoRA rank 32: only ~0.1% of parameters updated
""")

## G. Save Results to Drive

In [ ]:
SAVE_TO_DRIVE = False  # Set True to save checkpoint + results to Google Drive

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

    import shutil
    dest = '/content/drive/MyDrive/tinker-rl-submission/'
    os.makedirs(dest, exist_ok=True)

    # Copy checkpoint
    if os.path.exists('atropos/checkpoints/gsm8k_qwen8b/'):
        shutil.copytree('atropos/checkpoints/gsm8k_qwen8b/', f'{dest}/checkpoint/', dirs_exist_ok=True)

    # Copy results
    import glob
    for f in glob.glob('reports/final/gsm8k_heldout_*.json'):
        shutil.copy2(f, dest)

    print(f'Saved to {dest}')
else:
    print('Drive save skipped. Set SAVE_TO_DRIVE=True to enable.')

---

## Citation

```bibtex
@article{grpo_agentic_llm_2026,
  title={Group Relative Policy Optimization for Agentic LLM Fine-Tuning},
  author={PES University MTech Capstone Group 6},
  year={2026},
  note={Available at \url{https://github.com/pes-llm-research/tinker-rl-lab}}
}
```

## License

Apache 2.0 — See [LICENSE](https://github.com/pes-llm-research/tinker-rl-lab/blob/main/LICENSE)